[Tour itinerary](README.md) · [t02 · The solar neighborhood](t02_the_solar_neighborhood.ipynb) →
<!--nav-->

# t01 · One sky, many eyes

**The big idea:** there is no such thing as *the* picture of an astronomical object. A
telescope is tuned to a window of the electromagnetic spectrum, and each window is produced by
**different physics** — so pointing seven different instruments at the same patch of sky gives
you seven genuinely different measurements, not seven copies of one photo. Learning to read
them is learning what the object *is doing*.

We'll prove it on the best possible target, then take a quick gallery tour of the sky's
greatest hits — every image pulled live from public archives through
[SkyView](https://skyview.gsfc.nasa.gov/), NASA's "virtual observatory" that serves cutouts
from dozens of all-sky surveys.

**The target: the Crab Nebula (M1).** On 4 July 1054, Chinese court astronomers recorded a
"guest star" in Taurus bright enough to see in daylight for three weeks. It was a supernova —
a massive star's core collapsing — 6,500 light-years away. What's left today is a cloud of
shredded star guts expanding at ~1,500 km/s, powered from inside by the collapsed core: a
**pulsar**, a city-sized ball of neutrons spinning **30 times per second**. The Crab is the
one object bright in *every* band from radio to gamma rays, which is why half of high-energy
astronomy historically calibrated against it.

📖 *Resources:* [Crab Nebula](https://en.wikipedia.org/wiki/Crab_Nebula) ·
[SN 1054](https://en.wikipedia.org/wiki/SN_1054) ·
[primer p3](../primer/p3_how_we_observe.ipynb) (the spectrum & telescopes) ·
[SkyView surveys list](https://skyview.gsfc.nasa.gov/current/cgi/survey.pl)

In [ ]:
import os
from pathlib import Path

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "4")

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize, PercentileInterval
from astroquery.skyview import SkyView

CACHE = Path("../data/cache/tour/skyview")
CACHE.mkdir(parents=True, exist_ok=True)


def get_image(position, survey, width_deg=0.4, pixels=400):
    """Fetch a SkyView cutout, caching the FITS locally so re-runs are offline."""
    slug = f"{position}_{survey}_{width_deg}".replace(" ", "_").replace("/", "-")
    path = CACHE / f"{slug}.fits"
    if not path.exists():
        imgs = SkyView.get_images(position=position, survey=[survey],
                                  width=width_deg * u.deg, height=width_deg * u.deg,
                                  pixels=pixels)
        imgs[0].writeto(path)
    with fits.open(path) as hdul:
        return hdul[0].data.astype(float)


def show(ax, data, title, subtitle="", stretch="zscale", cmap="inferno"):
    if stretch == "zscale":
        lo, hi = ZScaleInterval().get_limits(data)
        norm = None
        ax.imshow(data, vmin=lo, vmax=hi, origin="lower", cmap=cmap)
    else:  # asinh: for images with a huge dynamic range
        norm = ImageNormalize(data, interval=PercentileInterval(99.8), stretch=AsinhStretch(0.1))
        ax.imshow(data, norm=norm, origin="lower", cmap=cmap)
    ax.set_title(f"{title}\n{subtitle}" if subtitle else title, fontsize=9)
    ax.set_xticks([]), ax.set_yticks([])

## 1. See it: the Crab through seven windows

Same object, same 0.25° patch of sky (except the last panel — more on that below). Watch how
the *shape* changes, because each band is emitted by a different component of the wreck:

In [ ]:
# (survey, label, what physically glows in this band)
CRAB_WINDOWS = [
    ("NVSS",          "Radio · 1.4 GHz (NVSS)",        "synchrotron: e\u207b spiraling in the magnetic field"),
    ("2MASS-K",       "Near-infrared · 2.2 \u00b5m (2MASS)", "stars, plus the synchrotron glow reaching the near-IR"),
    ("WISE 22",       "Mid-infrared · 22 \u00b5m (WISE)",    "warm dust condensed from the ejecta"),
    ("DSS2 Red",      "Optical · ~0.65 \u00b5m (DSS2)",      "glowing gas filaments: the shredded star"),
    ("GALEX Near UV", "Ultraviolet · 0.23 \u00b5m (GALEX)",  "the hottest gas and synchrotron core"),
    ("RASS-Cnt Soft", "X-ray · ~0.25 keV (ROSAT)",     "the pulsar wind: particles at ~10\u2077 K equivalent"),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 8.8))
for ax, (survey, label, physics) in zip(axes.flat, CRAB_WINDOWS):
    show(ax, get_image("M1", survey, width_deg=0.25), label, physics, stretch="asinh")
fig.suptitle("The Crab Nebula: one object, six instruments", y=0.99)
fig.tight_layout()
fig.subplots_adjust(hspace=0.16)
plt.show()

In [ ]:
# The seventh window needs its own scale: in gamma rays the telescope can't focus,
# so the sharpest image physics allows of this *point-like* source is degrees wide.
fig, ax = plt.subplots(figsize=(4.6, 4.6))
show(ax, get_image("M1", "Fermi 5", width_deg=10, pixels=300),
     "Gamma rays \u00b7 >1 GeV (Fermi)", "10\u00b0 field \u2014 25\u00d7 wider than the panels above")
plt.show()

### How to read what you just saw

- **Radio** and **X-ray** both show the *synchrotron nebula* — electrons accelerated to near
  light-speed by the pulsar, spiraling in the magnetic field. The X-ray version is smaller
  because X-ray-bright electrons lose their energy in years, so they can't drift far from the
  pulsar before fading; radio electrons last millennia and fill the whole bubble. **The size
  difference between two images is a particle-physics measurement.**
- **Optical** shows something else entirely: the filaments of actual stellar material
  (hydrogen, helium, sulfur...) glowing like a neon sign where the wind slams into it.
- **Near-IR** shows both at once: the field fills with ordinary stars (every deep image is a
  stack of foreground and background), while the nebula's synchrotron glow persists — the
  same emission as the radio, five hundred times shorter in wavelength.
- **Gamma rays**: the Crab is the brightest steady GeV source in the sky, yet the image is a
  blur, because gamma-ray photons cannot be focused by any mirror — they are detected almost
  one by one, by the electron showers they cause. Angular resolution is a resource that
  varies by *orders of magnitude* across the spectrum ([primer p3](../primer/p3_how_we_observe.ipynb)).

### The data behind the pictures

Images are only one of the data products this object generates. For the Crab, the working
astronomer's menu is:

| data type | what it is for the Crab | where it lives |
|---|---|---|
| **images** (above) | morphology per band; expansion measured by comparing epochs decades apart | SkyView, [MAST](https://mast.stsci.edu/), mission archives |
| **time series / timing** | the pulsar's 33 ms tick, timed for 50+ years; it spins *down* by ~36 ns/day, and that lost energy is exactly what lights the nebula | [Jodrell Bank Crab ephemeris](https://www.jb.man.ac.uk/pulsar/crab.html) |
| **spectra** | which elements the filaments contain, their speeds (Doppler), temperatures | mission archives |
| **polarization** | maps the magnetic field structure (IXPE, 2022+) | [HEASARC](https://heasarc.gsfc.nasa.gov/) |
| **catalogs** | the Crab as one row among millions, per survey | VizieR, survey databases |

The course lives almost entirely in row two (time series). The tour's job is to show you the
whole menu.

## 2. The gallery: greatest hits, each in the band where it shines

Six more objects — six different *classes* of object — each pulled from the archive in a
band chosen to show off what it is. Each caption names the tour stop that will cover it.

In [ ]:
GALLERY = [
    ("M42",         "DSS2 Red",      1.2, "zscale", "M42 Orion Nebula \u00b7 optical",
     "a stellar nursery: gas lit by newborn stars (stop t05)"),
    ("M42",         "WISE 12",       1.2, "asinh",  "M42 again \u00b7 mid-infrared",
     "same field: now the *dust* glows, revealing the cocoon (t05)"),
    ("M13",         "DSS2 Red",      0.4, "asinh",  "M13 Hercules cluster \u00b7 optical",
     "~300,000 stars, all ~12 billion years old (t02/t07)"),
    ("M57",         "DSS2 Red",      0.1, "zscale", "M57 Ring Nebula \u00b7 optical",
     "a sun-like star's gentle death: ejected outer layers (t04)"),
    ("M31",         "DSS2 Red",      3.0, "asinh",  "M31 Andromeda \u00b7 optical",
     "the nearest big galaxy: a trillion stars, 2.5 Mly away (t08)"),
    ("Cygnus Loop", "RASS-Cnt Soft", 4.0, "zscale", "Cygnus Loop \u00b7 X-ray",
     "a 20,000-year-old supernova blast wave, 3\u00b0 across (t04)"),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 9))
for ax, (obj, survey, w, stretch, label, caption) in zip(axes.flat, GALLERY):
    show(ax, get_image(obj, survey, width_deg=w), label, caption, stretch=stretch)
fig.suptitle("Six object classes, six fields, all live from the archive", y=0.985)
fig.tight_layout()
plt.show()

Things worth staring at:

- **The M42 pair** is the tour's thesis in two panels: in the optical you see the glowing gas
  *between* the dust; at 22 µm you see the dust itself, warm from the stars forming inside
  it. Neither is "what Orion really looks like." Both are.
- **M13**: every dot is a star, and they formed *together* — clusters are nature's controlled
  experiment, which is why stellar physics is calibrated on them.
- **M57 vs the Cygnus Loop** are the two ways stars die: a quiet shrug of ejected layers
  (sun-like stars) versus a blast wave still slamming through interstellar gas 20,000 years
  after a massive star exploded — 40× bigger on the sky, visible only in X-rays because the
  shocked gas is at millions of kelvin.

## 3. The frontier, and what's next

The frontier of "looking at the sky in many bands" is no longer building the images — it's
**time and simultaneity**:

- **The sky as a movie.** Rubin Observatory's LSST **began its ten-year survey in June
  2026**: the whole southern sky every ~3 nights, generating ~10 million alerts *per night*
  for anything that moved or changed. The bottleneck has flipped from photons to attention —
  deciding, within minutes and by machine, which of tonight's alerts deserve a spectrograph
  pointed at them.
- **Multi-messenger coincidence.** The 2017 neutron-star merger GW170817 was seen in
  gravitational waves, then gamma rays 1.7 s later, then in every electromagnetic band for
  weeks. Making that routine — catching counterparts before they fade — is arguably the
  hottest observational problem in the field (future stop t09).
- **New windows are still opening**: imaging X-ray polarization (IXPE) began in 2022, and the
  square-kilometre radio arrays are under construction.

**Where a hobbyist fits:** exactly at the attention bottleneck. Surveys publish more alerts
than professionals can vet, and the classification/vetting layer is open — that is the same
seam [IDEAS.md](../IDEAS.md) identified from the transit side. The follow-up stops of this
tour build toward being useful there.

*Everything above cached under `data/cache/tour/skyview/` (~15 MB); re-runs are offline.*

---
Next stop: [t02 · The solar neighborhood](t02_the_solar_neighborhood.ipynb) — from pictures
to a census: what Gaia knows about every star within 220 light-years.